# Vast.ai · 30-epoch Context–Membership study

Edit the **settings cell**, then use **Run All**. This notebook prepares the source/environment/data, checks the GPU, and runs the selected jobs. It uses the branch's existing launcher: 30 epochs, global batch 4, validation selection, and **one Hugging Face upload after each completed run**.

### Before renting

1. On Vast.ai, choose **Templates → Recommended → NVIDIA CUDA**, then rent an instance and open **Jupyter** through its Open/Instance Portal link. [Vast template quick start](https://docs.vast.ai/guides/templates/quickstart), [Jupyter access](https://docs.vast.ai/guides/instances/connect/jupyter).
2. Planning target: **one NVIDIA GPU with 48 GB VRAM or more, 32 GB allocated system RAM, at least 4 CPU cores, and 200 GB disk** for all 13 runs. These are conservative planning targets, not measured minimums; the full-batch probe below checks the actual fit. Disk must also hold the downloaded ZIP, extracted data, environment/cache, and local checkpoints.
3. Choose a host with **NVIDIA driver 580 or newer**. The lockfile uses a CUDA 13 PyTorch build; the template's CUDA label alone does not establish host-driver compatibility. [NVIDIA compatibility](https://docs.nvidia.com/deploy/cuda-compatibility/minor-version-compatibility.html).
4. Upload this notebook. The source branch must already be pushed, or upload the current source folder and set `REPO_DIR` to it. Setup does not push, pull over, or reset an existing checkout.
5. Supply a write-enabled `HF_TOKEN` through your instance environment, or enter it in the masked prompt. Do not paste tokens into notebook source. The destination is **public**: `Last-Bullet/vizwiz-cm-aciids-2027`.

The training worker runs independently of the browser connection. **Run All again attaches to an existing worker**. Interrupting the monitoring cell stops monitoring; it does not stop the worker. Destroying the instance loses local-only unfinished work: there are no automatic intermediate-epoch uploads. [Vast storage lifecycle](https://docs.vast.ai/guides/instances/storage/types).

## 1. Settings

Leave `RUN_IDS = []` for all 13 runs. FTC, CM-Shared, and CM-Separate each use seeds 42/123/456; the three component ablations and capacity control use seed 42. Keep `PROCESSES` consistent across this study and when resuming.

The dataset ZIP is downloaded directly from your Hugging Face dataset. A completed extraction is reused and pinned to its recorded source commit.

In [ ]:
from pathlib import Path

WORK_DIR = "/workspace/cm-study"
REPO_DIR = ""  # Existing source folder; blank uses this notebook's folder if valid, otherwise clones below WORK_DIR.
GIT_URL = "https://github.com/DPTPhatUS/VizWiz-VQA-Grounding.git"
GIT_REF = "exp/context-membership"  # Branch/tag to clone; existing source is reused without pulling.

DATASET_REPO = "Last-Bullet/VizWiz-VQA-Grounding"
DATASET_FILE = "vizwiz.zip"
DATASET_REVISION = "main"  # A commit SHA can pin a new extraction explicitly.
HF_REPO = "Last-Bullet/vizwiz-cm-aciids-2027"
HF_PREFIX = "matched-30ep-v1"

PROCESSES = 1
RUN_IDS = []  # Example: ["cm_shared_s42", "cm_shared_s123", "cm_shared_s456"]
AUTO_RESTORE_FROM_HF = True
RUN_OFFLINE_TESTS = True
RUN_GPU_PROBE = True
LAUNCH_TRAINING = True  # False prepares/checks/previews without starting training.

# Conservative guards. Lower only after considering the full-batch probe and actual instance allocation.
MIN_FREE_VRAM_GIB = 40
MIN_AVAILABLE_RAM_GIB = 24
MIN_FREE_DISK_GIB = 16  # Additional campaign storage is checked after dataset preparation.
MONITOR_SECONDS = 10


## 2. Source and isolated Python environment

The notebook kernel can stay on the template's Python. Training runs in a separate Python 3.12 environment created from `uv.lock`, so no kernel restart is needed. Existing source and datasets are never overwritten. Setup cells skip heavy work when this workspace already has an active worker.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone

NOTEBOOK_DIR = Path.cwd().resolve()
WORK = Path(WORK_DIR).expanduser().resolve()
WORK.mkdir(parents=True, exist_ok=True)
JOB_RECORD = WORK / "vast-job.json"

identity = dict(work_dir=str(WORK), repo_dir=REPO_DIR, git_url=GIT_URL, git_ref=GIT_REF,
                dataset_repo=DATASET_REPO, dataset_file=DATASET_FILE, dataset_revision=DATASET_REVISION,
                hf_repo=HF_REPO, hf_prefix=HF_PREFIX, processes=PROCESSES, runs=RUN_IDS)
SETTINGS_SHA = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text())

def write_record(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n")
    os.replace(temporary, path)

def job_running(record):
    try:
        arguments = Path(f"/proc/{int(record['pid'])}/cmdline").read_bytes().split(b"\0")
    except (FileNotFoundError, ProcessLookupError, KeyError, ValueError):
        return False
    return b"experiments.vast" in arguments and b"worker" in arguments and os.fsencode(record["config"]) in arguments

previous_job = read_json(JOB_RECORD) if JOB_RECORD.exists() else None
LIVE_JOB = bool(previous_job and job_running(previous_job))
if LIVE_JOB and previous_job["settings_sha"] != SETTINGS_SHA:
    raise RuntimeError("This workspace has an active job with different settings. Keep its settings to monitor it, or use a separate workspace after stopping it.")

def run(command):
    print("Running:", " ".join(str(part) for part in command), flush=True)
    subprocess.run([str(part) for part in command], cwd=REPO, check=True)

if LIVE_JOB:
    REPO = Path(previous_job["repo_dir"])
    PYTHON = Path(previous_job["python"])
    print(f"Active worker PID {previous_job['pid']}: setup will be skipped; monitoring will reconnect.")
else:
    if PROCESSES < 1 or 4 % PROCESSES:
        raise ValueError("PROCESSES must divide the fixed global batch size of 4")
    if REPO_DIR:
        REPO = Path(REPO_DIR).expanduser().resolve()
    elif (NOTEBOOK_DIR / "train.py").is_file() and (NOTEBOOK_DIR / "pyproject.toml").is_file():
        REPO = NOTEBOOK_DIR
    else:
        REPO = WORK / "VizWiz-VQA-Grounding"
    if not (REPO / "train.py").is_file():
        if REPO.exists() and any(REPO.iterdir()):
            raise RuntimeError(f"{REPO} is nonempty but is not a source checkout; set REPO_DIR correctly")
        REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--single-branch", "--branch", GIT_REF, GIT_URL, str(REPO)], check=True)
    required = ["pyproject.toml", "uv.lock", "configs/high_priority.json", "experiments/vast.py", "experiments/hub.py"]
    missing = [name for name in required if not (REPO / name).is_file()]
    if missing:
        raise RuntimeError(f"Source is outdated or incomplete: {missing}. Push the current branch before cloning, or upload the current source folder.")
    MANIFEST = REPO / "configs/high_priority.json"
    manifest = read_json(MANIFEST)
    expected = {f"{model}_s{seed}" for model in ("ftc", "cm_shared", "cm_separate") for seed in (42, 123, 456)}
    expected |= {"context_only_s42", "membership_only_s42", "cm_noaux_s42", "ftc_capacity_s42"}
    if manifest["common"]["num_epochs"] != 30 or {item["id"] for item in manifest["runs"]} != expected:
        raise RuntimeError("Source does not contain the current 13-run / 30-epoch preset; update the source checkout")
    if len(set(RUN_IDS)) != len(RUN_IDS) or (RUN_IDS and not set(RUN_IDS) <= expected):
        raise ValueError("RUN_IDS contains duplicates or unknown IDs")
    if not (REPO / ".git").exists():
        print("Using uploaded source; provenance will also record its source hash.")
    else:
        subprocess.run(["git", "-C", str(REPO), "log", "-1", "--format=%h %s"], check=True)
    os.environ.update(HF_HOME=str(WORK / "cache/huggingface"), UV_CACHE_DIR=str(WORK / "cache/uv"),
                      MPLCONFIGDIR=str(WORK / "cache/matplotlib"), MPLBACKEND="Agg", OMP_NUM_THREADS="1",
                      PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    uv = shutil.which("uv")
    if uv is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "uv"], check=True)
        candidates = [shutil.which("uv"), str(Path(sys.executable).parent / "uv"), str(Path.home() / ".local/bin/uv")]
        uv = next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)
    if uv is None:
        raise RuntimeError("uv installation did not expose an executable; install uv and rerun this cell")
    run([uv, "python", "install", "3.12"])
    run([uv, "sync", "--frozen", "--python", "3.12"])
    PYTHON = REPO / ".venv/bin/python"
    print("Training interpreter:", PYTHON)


## 3. Hardware and Hugging Face access

Hardware checks use the isolated environment. The token is inherited by child processes, never passed as a command argument or written to job configuration. Write access and dataset-file access are checked before the large download.

In [ ]:
if LIVE_JOB:
    print("Active worker: hardware/authentication setup skipped.")
else:
    run([PYTHON, "-m", "experiments.vast", "hardware", "--workspace", WORK, "--processes", PROCESSES,
         "--min-vram-gib", MIN_FREE_VRAM_GIB, "--min-ram-gib", MIN_AVAILABLE_RAM_GIB,
         "--min-disk-gib", MIN_FREE_DISK_GIB, "--report", WORK / "hardware.json"])
    if not os.environ.get("HF_TOKEN", "").strip():
        from getpass import getpass
        token = getpass("Hugging Face write token (input hidden): ").strip()
        if not token:
            raise RuntimeError("A write-enabled Hugging Face token is required")
        os.environ["HF_TOKEN"] = token
        del token
    run([PYTHON, "-m", "experiments.vast", "auth", "--repo-id", HF_REPO,
         "--dataset-repo", DATASET_REPO, "--filename", DATASET_FILE, "--revision", DATASET_REVISION,
         "--report", WORK / "access.json"])


## 4. Download and validate the dataset

`vizwiz.zip` is downloaded with Hugging Face's resumable cache into `downloads/`, extracted into a fresh staging directory, and then moved into place. Archive paths are checked; the source commit and archive SHA-256 are recorded. A wrapping `vizwiz/` directory is detected automatically. Repeated setup reuses the same pinned extraction.

In [ ]:
if LIVE_JOB:
    DATA_ROOT = Path(read_json(previous_job["config"])["data_root"])
    print("Active worker: using its dataset", DATA_ROOT)
else:
    run([PYTHON, "-m", "experiments.vast", "data", "--repo-id", DATASET_REPO,
         "--filename", DATASET_FILE, "--revision", DATASET_REVISION,
         "--download-dir", WORK / "downloads", "--extract-dir", WORK / "data",
         "--report", WORK / "dataset.json"])
    DATA_ROOT = Path(read_json(WORK / "dataset.json")["data_root"])
    check_all_splits = "from preflight import validate_data; import json,sys; print(json.dumps(validate_data(sys.argv[1], ('train','val','test'))))"
    run([PYTHON, "-c", check_all_splits, DATA_ROOT])
    OUTPUT_ROOT = WORK / "outputs/revision"
    selected = [item for item in manifest["runs"] if not RUN_IDS or item["id"] in RUN_IDS]
    new_runs = sum(not (OUTPUT_ROOT / item["id"] / "latest.pt").is_file() for item in selected)
    required_gib = 6 * new_runs + 12
    free_gib = shutil.disk_usage(WORK).free / 2**30
    if free_gib < required_gib:
        raise RuntimeError(f"Only {free_gib:.1f} GiB free; allow about {required_gib} GiB for retained checkpoints and temporary saving. Select fewer runs or use a larger disk.")
    print(f"Dataset ready: {DATA_ROOT}; free disk {free_gib:.1f} GiB; selected runs {len(selected)}")


## 5. Tests, full-batch fit check, and command preview

The probe uses full CM-Separate at the matched per-GPU batch and completes **two successful Adam steps**, including optimizer-state memory. It never changes the scientific batch size, freezes extra layers, or substitutes a smaller model. An OOM stops setup before the campaign. Probe weights are discarded; real runs start with their own seeds. For multiple GPUs this probes each device separately, not NCCL communication overhead.

In [ ]:
if LIVE_JOB:
    print("Active worker: tests and memory probe skipped.")
else:
    if RUN_OFFLINE_TESTS:
        run([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"])
    if RUN_GPU_PROBE:
        run([PYTHON, "-m", "experiments.vast", "probe", "--data-root", DATA_ROOT,
             "--processes", PROCESSES, "--report", WORK / "gpu-probe.json"])
    preview = [PYTHON, "-m", "experiments.launch", "--data-root", DATA_ROOT,
               "--output-root", OUTPUT_ROOT, "--processes", PROCESSES,
               "--hf-repo", HF_REPO, "--hf-prefix", HF_PREFIX, "--dry-run"]
    if RUN_IDS:
        preview += ["--runs", *RUN_IDS]
    run(preview)


## 6. Start or reattach to the study

A new worker uses the existing launcher for each run. It resumes local `latest.pt` files; with `AUTO_RESTORE_FROM_HF`, it restores a remote run when the local folder is empty. Current-format identities, hashes, process count, and recipe checks still apply. It never deletes local runs or resets a remote destination.

Only after a run finishes training **and test evaluation** does it upload `latest.pt`, `best.pt`, and records/metrics. Finished runs are summarized; a local records archive includes prediction PNGs but excludes checkpoint payloads. The worker's log and status live under `WORK_DIR/jobs/`.

In [ ]:
PROCESS = None
JOB = previous_job if LIVE_JOB else None
if LIVE_JOB:
    print(f"Reattaching to PID {JOB['pid']}")
elif not LAUNCH_TRAINING:
    print("Setup and preview complete. LAUNCH_TRAINING=False: no training job started.")
else:
    session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    SESSION = WORK / "jobs" / session_id
    SESSION.mkdir(parents=True)
    configuration = dict(session_id=session_id, session_dir=str(SESSION), manifest=str(MANIFEST),
                         data_root=str(DATA_ROOT), output_root=str(OUTPUT_ROOT), processes=PROCESSES,
                         runs=RUN_IDS, hf_repo=HF_REPO, hf_prefix=HF_PREFIX,
                         auto_restore=AUTO_RESTORE_FROM_HF, upload=True, device="cuda")
    config_file = SESSION / "config.json"
    write_record(config_file, configuration)
    log_file = SESSION / "worker.log"
    command = [str(PYTHON), "-u", "-m", "experiments.vast", "worker", "--config", str(config_file)]
    with log_file.open("ab", buffering=0) as log:
        PROCESS = subprocess.Popen(command, cwd=REPO, stdin=subprocess.DEVNULL,
                                   stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy(), start_new_session=True)
    JOB = dict(pid=PROCESS.pid, config=str(config_file), repo_dir=str(REPO), python=str(PYTHON),
               session_dir=str(SESSION), log=str(log_file), settings_sha=SETTINGS_SHA)
    write_record(JOB_RECORD, JOB)
    print(f"Started worker PID {PROCESS.pid}. Full log: {log_file}")


## 7. Monitor

This cell waits for the complete selected queue. It shows epoch summaries and upload confirmations; the full raw log stays on disk. Closing the browser does not depend on this cell staying connected. If the kernel restarts, use Run All to reattach. To stop the worker explicitly, use the optional stop cell at the end; current-epoch work may need to be repeated.

In [ ]:
FINAL_STATUS = None
if JOB:
    log_path = Path(JOB["log"])
    status_path = Path(JOB["session_dir"]) / "status.json"
    offset, pending = 0, ""
    try:
        while True:
            if log_path.exists():
                with log_path.open(errors="replace") as log:
                    log.seek(offset)
                    pending += log.read().replace("\r", "\n")
                    offset = log.tell()
                lines = pending.split("\n")
                pending = lines.pop()
                for line in lines:
                    text = line.strip()
                    if text.startswith(("Starting ", "Resuming ", "Restoring ")):
                        print(text, flush=True)
                    elif text.startswith("{"):
                        try:
                            row = json.loads(text)
                        except json.JSONDecodeError:
                            continue
                        if "train_loss" in row and "epoch" in row:
                            print(f"Epoch {row['epoch']:2d}/30 | loss {row['train_loss']:.5f} | val mIoU {row.get('validation_miou', float('nan')):.5f}", flush=True)
                        elif "huggingface_final" in row:
                            result = row["huggingface_final"]
                            print(f"Hub backup confirmed: {result['path']} @ {result.get('commit', 'unchanged')}", flush=True)
            status = read_json(status_path) if status_path.exists() else {}
            if status.get("state") == "complete":
                FINAL_STATUS = status
                print("All selected runs completed; summaries and records archive are ready.")
                break
            if status.get("state") == "failed" or not job_running(JOB):
                tail = log_path.read_text(errors="replace").splitlines()[-35:] if log_path.exists() else []
                print("\n".join(tail))
                raise RuntimeError(f"Worker stopped: {status.get('error', 'no completion record')}. Full log: {log_path}. Fix the cause and Run All to resume.")
            time.sleep(max(1, MONITOR_SECONDS))
    except KeyboardInterrupt:
        print(f"Monitoring interrupted. Worker PID {JOB['pid']} continues; Run All reattaches.")
        raise


## 8. Results and downloads

Automatic Hub backups contain checkpoints and run/test records. The local records ZIP additionally includes native prediction PNGs. For an unfinished run, stop its worker before using the manual backup command printed below. The setup's token must remain available when retrying a failed upload.

In [ ]:
if FINAL_STATUS:
    from IPython.display import FileLink, display
    summary_path = Path(FINAL_STATUS["summary"])
    summary = read_json(summary_path)
    for name, values in summary["by_configuration"].items():
        score = values["mean_iou"]
        deviation = "n/a" if score["sample_std"] is None else f"{score['sample_std']:.5f}"
        print(f"{name:24s} mIoU mean={score['mean']:.5f}, sample SD={deviation}")
    print("Missing study runs:", summary["missing_runs"])
    for path in [summary_path, summary_path.with_name("runs.csv"), Path(FINAL_STATUS["records_archive"]), Path(JOB["log"])]:
        print(path)
        display(FileLink(os.path.relpath(path, NOTEBOOK_DIR)))
elif not JOB:
    print("No training launched. Change LAUNCH_TRAINING to True when ready.")

print("Manual backup of a stopped, unfinished run (replace RUN_ID):")
print(f"{PYTHON} -m experiments.hub upload --run-dir {WORK / 'outputs/revision/RUN_ID'} --path-in-repo {HF_PREFIX}/RUN_ID")


## Optional: explicitly stop the worker

This cell does nothing during Run All. To stop training, interrupt the monitoring cell, change `STOP_WORKER` below, and run **only this cell**. It stops the worker and its training process group. The last completed local epoch is the resume point; unfinished progress is not automatically uploaded. Do not destroy the instance until any needed manual backup is confirmed.

In [ ]:
STOP_WORKER = False
if STOP_WORKER:
    import signal
    record = read_json(JOB_RECORD)
    if job_running(record):
        os.killpg(int(record["pid"]), signal.SIGTERM)
        print("Stop requested. Wait for the worker to exit before a manual backup or resume.")
    else:
        print("No matching worker is running.")
